In [1]:
"""
Generate a realistic, balanced synthetic Uttarakhand disaster incident dataset.

Outputs
-------
1. disaster_incidents_balanced.csv
2. class_distribution.png
3. people_affected_distribution.png
4. injured_people_distribution.png
5. disaster_type_distribution.png
6. correlation_matrix.png

Dataset size
------------
P1: 2,000
P2: 2,000
P3: 2,000
Total: 6,000

Method
------
Priority is selected first, followed by class-conditional probabilistic
generation of correlated incident features.

Overlapping latent severity distributions deliberately create ambiguous
boundary cases. Priority is not generated with a single deterministic rule.
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

RANDOM_SEED = 42
RECORDS_PER_CLASS = 2_000
OUTPUT_FILE = "disaster_incidents_balanced.csv"
OUTPUT_DIRECTORY = Path(".")

rng = np.random.default_rng(RANDOM_SEED)

PRIORITIES = ["P1", "P2", "P3"]

DISASTER_TYPES = [
    "Flood",
    "Cloudburst",
    "Landslide",
    "Earthquake",
]

INCIDENT_TYPES = [
    "Trapped People",
    "Injured Person",
    "Missing Person",
    "Flooded Road",
    "Damaged House",
    "Evacuation Request",
    "Waterlogging",
    "Infrastructure Damage",
]

ROAD_STATUSES = [
    "Open",
    "Partially Blocked",
    "Blocked",
]

HOUSE_DAMAGE_LEVELS = [
    "None",
    "Minor",
    "Moderate",
    "Severe",
]

MEDICAL_SEVERITIES = [
    "None",
    "Low",
    "Moderate",
    "Critical",
]

EXPECTED_COLUMNS = [
    "incident_id",
    "disaster_type",
    "incident_type",
    "people_affected",
    "people_trapped",
    "injured_people",
    "missing_people",
    "children_affected",
    "elderly_affected",
    "water_level_m",
    "flooded_area_percent",
    "road_status",
    "house_damage",
    "evacuation_route_available",
    "power_available",
    "communication_available",
    "medical_required",
    "medical_severity",
    "ambulance_required",
    "latitude",
    "longitude",
    "nearest_hospital_km",
    "nearest_shelter_km",
    "nearest_rescue_team_km",
    "population_density",
    "priority",
]


# Approximate district/response-region centers in Uttarakhand.
# Random spatial jitter creates synthetic coordinates around these centers.
UTTARAKHAND_REGIONS = {
    "Dehradun": (30.3165, 78.0322, 0.23),
    "Haridwar": (29.9457, 78.1642, 0.14),
    "Nainital": (29.3919, 79.4542, 0.11),
    "Udham Singh Nagar": (28.9610, 79.5154, 0.11),
    "Pauri Garhwal": (30.1471, 78.7745, 0.08),
    "Tehri Garhwal": (30.3782, 78.4803, 0.07),
    "Chamoli": (30.4046, 79.3315, 0.06),
    "Rudraprayag": (30.2844, 78.9811, 0.05),
    "Uttarkashi": (30.7268, 78.4354, 0.05),
    "Almora": (29.5892, 79.6467, 0.04),
    "Pithoragarh": (29.5829, 80.2182, 0.03),
    "Bageshwar": (29.8370, 79.7710, 0.02),
    "Champawat": (29.3362, 80.0910, 0.01),
}


# Different priorities can draw from boundary profiles. This intentionally
# introduces class overlap without corrupting labels randomly.
SEVERITY_PROFILES = {
    "P1": {
        "critical": 0.67,
        "high_boundary": 0.25,
        "moderate_exception": 0.08,
    },
    "P2": {
        "critical_boundary": 0.15,
        "high": 0.60,
        "moderate_boundary": 0.25,
    },
    "P3": {
        "high_exception": 0.08,
        "moderate_boundary": 0.25,
        "moderate": 0.67,
    },
}


def clip(value, minimum, maximum):
    """Clip a numeric value to an inclusive range."""
    return min(max(value, minimum), maximum)


def bounded_int(value, minimum, maximum):
    """Round and clip a value as an integer."""
    return int(clip(round(value), minimum, maximum))


def weighted_choice(options, probabilities):
    """Select one value using normalized probabilities."""
    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = probabilities / probabilities.sum()
    return rng.choice(options, p=probabilities)


# ---------------------------------------------------------------------------
# Latent severity and geography
# ---------------------------------------------------------------------------

def sample_latent_severity(priority):
    """
    Sample an overlapping latent severity.

    The intervals overlap substantially:
      P1 commonly high, but can have moderate overall impact when a specific
         medical or access condition makes the incident critical.
      P2 spans moderate-to-high conditions.
      P3 is commonly low-to-moderate, with environmental exceptions.
    """
    profile_name = weighted_choice(
        list(SEVERITY_PROFILES[priority].keys()),
        list(SEVERITY_PROFILES[priority].values()),
    )

    beta_parameters = {
        "critical": (7.5, 2.0),
        "high_boundary": (5.0, 3.0),
        "moderate_exception": (3.5, 4.5),
        "critical_boundary": (6.0, 2.8),
        "high": (4.5, 4.0),
        "moderate_boundary": (3.0, 5.0),
        "high_exception": (4.8, 3.2),
        "moderate": (2.0, 6.5),
    }

    alpha, beta = beta_parameters[profile_name]
    severity = float(rng.beta(alpha, beta))

    return severity, profile_name


def sample_region():
    """Select an Uttarakhand response region and produce a nearby coordinate."""
    region_names = list(UTTARAKHAND_REGIONS.keys())
    probabilities = [UTTARAKHAND_REGIONS[name][2] for name in region_names]

    region = weighted_choice(region_names, probabilities)
    center_lat, center_lon, _ = UTTARAKHAND_REGIONS[region]

    # Mountain districts cover wider areas than city centers.
    mountain_regions = {
        "Chamoli",
        "Rudraprayag",
        "Uttarkashi",
        "Pithoragarh",
        "Bageshwar",
        "Tehri Garhwal",
    }

    spread = 0.12 if region in mountain_regions else 0.07

    latitude = center_lat + rng.normal(0, spread)
    longitude = center_lon + rng.normal(0, spread)

    # Keep all synthetic coordinates within Uttarakhand's broad extent.
    latitude = clip(latitude, 28.75, 31.45)
    longitude = clip(longitude, 77.55, 81.05)

    return region, round(latitude, 6), round(longitude, 6)


def sample_population_density(region):
    """Generate plausible population density based on regional character."""
    high_density = {"Dehradun", "Haridwar", "Udham Singh Nagar"}
    medium_density = {"Nainital", "Almora", "Pauri Garhwal", "Champawat"}

    if region in high_density:
        value = rng.lognormal(mean=np.log(1800), sigma=0.65)
    elif region in medium_density:
        value = rng.lognormal(mean=np.log(650), sigma=0.65)
    else:
        value = rng.lognormal(mean=np.log(220), sigma=0.65)

    return bounded_int(value, 50, 10_000)


# ---------------------------------------------------------------------------
# Disaster and incident selection
# ---------------------------------------------------------------------------

def sample_disaster_type(region):
    """Select a disaster type with regional plausibility."""
    plains_regions = {"Haridwar", "Udham Singh Nagar"}
    mountain_regions = {
        "Chamoli",
        "Rudraprayag",
        "Uttarkashi",
        "Pithoragarh",
        "Bageshwar",
        "Tehri Garhwal",
    }

    if region in plains_regions:
        probabilities = [0.55, 0.15, 0.10, 0.20]
    elif region in mountain_regions:
        probabilities = [0.23, 0.25, 0.37, 0.15]
    else:
        probabilities = [0.32, 0.20, 0.28, 0.20]

    return weighted_choice(DISASTER_TYPES, probabilities)


def sample_incident_type(disaster_type, severity):
    """Choose an incident type based on disaster and latent severity."""
    base_weights = {
        "Flood": {
            "Trapped People": 0.14,
            "Injured Person": 0.06,
            "Missing Person": 0.06,
            "Flooded Road": 0.23,
            "Damaged House": 0.10,
            "Evacuation Request": 0.16,
            "Waterlogging": 0.19,
            "Infrastructure Damage": 0.06,
        },
        "Cloudburst": {
            "Trapped People": 0.18,
            "Injured Person": 0.10,
            "Missing Person": 0.10,
            "Flooded Road": 0.16,
            "Damaged House": 0.13,
            "Evacuation Request": 0.15,
            "Waterlogging": 0.08,
            "Infrastructure Damage": 0.10,
        },
        "Landslide": {
            "Trapped People": 0.18,
            "Injured Person": 0.10,
            "Missing Person": 0.10,
            "Flooded Road": 0.05,
            "Damaged House": 0.15,
            "Evacuation Request": 0.13,
            "Waterlogging": 0.02,
            "Infrastructure Damage": 0.27,
        },
        "Earthquake": {
            "Trapped People": 0.18,
            "Injured Person": 0.17,
            "Missing Person": 0.08,
            "Flooded Road": 0.01,
            "Damaged House": 0.26,
            "Evacuation Request": 0.13,
            "Waterlogging": 0.01,
            "Infrastructure Damage": 0.16,
        },
    }

    weights = base_weights[disaster_type].copy()

    if severity > 0.68:
        weights["Trapped People"] *= 1.7
        weights["Injured Person"] *= 1.5
        weights["Missing Person"] *= 1.5
        weights["Waterlogging"] *= 0.55
    elif severity < 0.30:
        weights["Waterlogging"] *= 1.6
        weights["Flooded Road"] *= 1.25
        weights["Trapped People"] *= 0.55
        weights["Missing Person"] *= 0.55

    return weighted_choice(
        list(weights.keys()),
        list(weights.values()),
    )


# ---------------------------------------------------------------------------
# Correlated feature generation
# ---------------------------------------------------------------------------

def generate_environment(disaster_type, incident_type, severity):
    """Generate correlated environmental and infrastructure conditions."""
    water_factor = {
        "Flood": 1.00,
        "Cloudburst": 0.82,
        "Landslide": 0.22,
        "Earthquake": 0.04,
    }[disaster_type]

    water_level = (
        water_factor
        * (0.20 + 4.35 * severity)
        * rng.uniform(0.72, 1.22)
    )

    if incident_type in {"Flooded Road", "Waterlogging"}:
        water_level += rng.uniform(0.15, 0.80)

    water_level = round(clip(water_level, 0, 5), 2)

    flooded_area = (
        water_factor
        * (5 + 88 * severity)
        * rng.uniform(0.70, 1.20)
    )

    if incident_type in {"Flooded Road", "Waterlogging"}:
        flooded_area += rng.uniform(5, 18)

    flooded_area = round(clip(flooded_area, 0, 100), 2)

    road_risk = severity
    if disaster_type in {"Landslide", "Cloudburst"}:
        road_risk += 0.14
    if incident_type in {"Flooded Road", "Infrastructure Damage"}:
        road_risk += 0.18

    road_noise = rng.normal(0, 0.15)
    road_score = road_risk + road_noise

    if road_score > 0.72:
        road_status = weighted_choice(
            ["Blocked", "Partially Blocked", "Open"],
            [0.68, 0.27, 0.05],
        )
    elif road_score > 0.40:
        road_status = weighted_choice(
            ["Blocked", "Partially Blocked", "Open"],
            [0.28, 0.55, 0.17],
        )
    else:
        road_status = weighted_choice(
            ["Blocked", "Partially Blocked", "Open"],
            [0.05, 0.24, 0.71],
        )

    damage_score = severity
    if disaster_type == "Earthquake":
        damage_score += 0.18
    elif disaster_type == "Landslide":
        damage_score += 0.10
    if incident_type == "Damaged House":
        damage_score += 0.23
    damage_score += rng.normal(0, 0.16)

    if damage_score > 0.85:
        house_damage = weighted_choice(
            ["Severe", "Moderate", "Minor", "None"],
            [0.63, 0.28, 0.07, 0.02],
        )
    elif damage_score > 0.55:
        house_damage = weighted_choice(
            ["Severe", "Moderate", "Minor", "None"],
            [0.20, 0.52, 0.23, 0.05],
        )
    elif damage_score > 0.28:
        house_damage = weighted_choice(
            ["Severe", "Moderate", "Minor", "None"],
            [0.04, 0.22, 0.55, 0.19],
        )
    else:
        house_damage = weighted_choice(
            ["Severe", "Moderate", "Minor", "None"],
            [0.01, 0.05, 0.32, 0.62],
        )

    route_probability = 0.92 - 0.57 * severity
    if road_status == "Blocked":
        route_probability -= 0.27
    elif road_status == "Partially Blocked":
        route_probability -= 0.10

    # Alternative routes occasionally exist despite a blocked main road.
    route_probability = clip(route_probability, 0.05, 0.98)
    evacuation_route_available = int(rng.random() < route_probability)

    power_probability = 0.95 - 0.66 * severity
    if house_damage == "Severe":
        power_probability -= 0.16
    power_available = int(rng.random() < clip(power_probability, 0.04, 0.99))

    communication_probability = 0.97 - 0.47 * severity
    if disaster_type in {"Landslide", "Cloudburst"}:
        communication_probability -= 0.08
    communication_available = int(
        rng.random() < clip(communication_probability, 0.08, 0.99)
    )

    return {
        "water_level_m": water_level,
        "flooded_area_percent": flooded_area,
        "road_status": road_status,
        "house_damage": house_damage,
        "evacuation_route_available": evacuation_route_available,
        "power_available": power_available,
        "communication_available": communication_available,
    }


def generate_human_impact(
    severity,
    profile_name,
    incident_type,
    population_density,
    infrastructure,
):
    """Generate internally consistent human-impact fields."""
    density_factor = np.log1p(population_density) / np.log1p(10_000)

    expected_affected = (
        2
        + 104 * (severity ** 1.45)
        + 34 * density_factor * severity
    )

    if incident_type == "Evacuation Request":
        expected_affected *= 1.25
    elif incident_type == "Waterlogging":
        expected_affected *= 0.68
    elif incident_type == "Injured Person":
        expected_affected *= 0.52

    people_affected = bounded_int(
        rng.gamma(shape=2.4, scale=max(expected_affected / 2.4, 0.5)),
        0,
        200,
    )

    # Ensure human-centered incident categories have at least one affected person.
    if incident_type in {
        "Trapped People",
        "Injured Person",
        "Missing Person",
        "Evacuation Request",
    }:
        people_affected = max(1, people_affected)

    blocked_factor = {
        "Open": 0.0,
        "Partially Blocked": 0.10,
        "Blocked": 0.24,
    }[infrastructure["road_status"]]

    no_route_factor = 0.18 * (
        1 - infrastructure["evacuation_route_available"]
    )

    trapped_rate = clip(
        0.01
        + 0.28 * severity
        + blocked_factor
        + no_route_factor,
        0,
        0.75,
    )

    if incident_type == "Trapped People":
        trapped_rate = clip(trapped_rate + 0.20, 0, 0.85)
    elif incident_type in {"Waterlogging", "Flooded Road"}:
        trapped_rate *= 0.45

    people_trapped = int(
        rng.binomial(min(people_affected, 100), trapped_rate)
    )

    if incident_type == "Trapped People" and people_affected > 0:
        people_trapped = max(1, people_trapped)

    injury_rate = clip(0.005 + 0.095 * severity, 0, 0.24)

    if incident_type == "Injured Person":
        injury_rate += 0.18
    if infrastructure["house_damage"] == "Severe":
        injury_rate += 0.08

    injured_people = int(
        rng.binomial(min(people_affected, 30), clip(injury_rate, 0, 0.55))
    )

    if incident_type == "Injured Person" and people_affected > 0:
        injured_people = max(1, injured_people)

    missing_rate = clip(0.004 + 0.075 * severity, 0, 0.20)
    if incident_type == "Missing Person":
        missing_rate += 0.20
    if infrastructure["communication_available"] == 0:
        missing_rate += 0.04

    missing_people = int(
        rng.binomial(min(people_affected, 30), clip(missing_rate, 0, 0.50))
    )

    if incident_type == "Missing Person" and people_affected > 0:
        missing_people = max(1, missing_people)

    # Children and elderly are demographic subsets of people affected.
    children_rate = rng.uniform(0.10, 0.30)
    elderly_rate = rng.uniform(0.07, 0.22)

    children_affected = int(
        rng.binomial(min(people_affected, 50), children_rate)
    )

    remaining_after_children = max(people_affected - children_affected, 0)
    elderly_affected = int(
        rng.binomial(min(remaining_after_children, 40), elderly_rate)
    )

    return {
        "people_affected": people_affected,
        "people_trapped": min(people_trapped, people_affected, 100),
        "injured_people": min(injured_people, people_affected, 30),
        "missing_people": min(missing_people, people_affected, 30),
        "children_affected": min(children_affected, people_affected, 50),
        "elderly_affected": min(
            elderly_affected,
            max(people_affected - children_affected, 0),
            40,
        ),
    }


def generate_medical(priority, severity, profile_name, human):
    """Generate medical fields from injuries and latent conditions."""
    injured = human["injured_people"]
    trapped = human["people_trapped"]
    elderly = human["elderly_affected"]
    children = human["children_affected"]

    medical_probability = (
        0.03
        + 0.62 * severity
        + 0.06 * min(injured, 5)
        + 0.015 * min(trapped, 10)
        + 0.005 * min(elderly + children, 20)
    )

    if injured > 0:
        medical_probability = max(medical_probability, 0.82)

    medical_required = int(
        rng.random() < clip(medical_probability, 0.02, 0.99)
    )

    if medical_required == 0:
        return {
            "medical_required": 0,
            "medical_severity": "None",
            "ambulance_required": 0,
        }

    medical_score = (
        0.52 * severity
        + 0.06 * min(injured, 6)
        + 0.012 * min(trapped, 15)
        + rng.normal(0, 0.13)
    )

    # Introduce realistic exceptions and boundary overlap.
    if priority == "P1" and profile_name == "moderate_exception":
        medical_score += rng.uniform(0.20, 0.42)

    if medical_score > 0.78:
        severity_probs = [0.04, 0.10, 0.29, 0.57]
    elif medical_score > 0.52:
        severity_probs = [0.04, 0.20, 0.56, 0.20]
    else:
        severity_probs = [0.05, 0.66, 0.25, 0.04]

    medical_severity = weighted_choice(
        ["None", "Low", "Moderate", "Critical"],
        severity_probs,
    )

    # A required medical response cannot have "None" severity.
    if medical_severity == "None":
        medical_severity = "Low"

    ambulance_probability = {
        "Low": 0.20,
        "Moderate": 0.62,
        "Critical": 0.94,
    }[medical_severity]

    if injured >= 3:
        ambulance_probability += 0.10

    ambulance_required = int(
        rng.random() < clip(ambulance_probability, 0, 0.99)
    )

    return {
        "medical_required": 1,
        "medical_severity": medical_severity,
        "ambulance_required": ambulance_required,
    }


def generate_distances(region, severity, road_status):
    """Generate plausible service distances in Uttarakhand terrain."""
    urban_regions = {"Dehradun", "Haridwar", "Udham Singh Nagar"}
    semi_urban_regions = {"Nainital", "Almora", "Pauri Garhwal"}

    if region in urban_regions:
        hospital_base = rng.gamma(2.0, 2.2)
        shelter_base = rng.gamma(2.0, 1.5)
        rescue_base = rng.gamma(2.0, 2.0)
    elif region in semi_urban_regions:
        hospital_base = rng.gamma(2.5, 3.0)
        shelter_base = rng.gamma(2.3, 2.3)
        rescue_base = rng.gamma(2.5, 3.0)
    else:
        hospital_base = rng.gamma(3.0, 4.0)
        shelter_base = rng.gamma(2.7, 3.0)
        rescue_base = rng.gamma(3.0, 3.8)

    access_penalty = {
        "Open": 1.00,
        "Partially Blocked": 1.15,
        "Blocked": 1.32,
    }[road_status]

    # Report effective operational distances, which rise when access is poor.
    nearest_hospital = clip(hospital_base * access_penalty, 0.5, 30)
    nearest_shelter = clip(shelter_base * access_penalty, 0.2, 20)
    nearest_rescue_team = clip(rescue_base * access_penalty, 0.5, 30)

    return {
        "nearest_hospital_km": round(nearest_hospital, 2),
        "nearest_shelter_km": round(nearest_shelter, 2),
        "nearest_rescue_team_km": round(nearest_rescue_team, 2),
    }


# ---------------------------------------------------------------------------
# Record generation
# ---------------------------------------------------------------------------

def generate_record(incident_number, priority):
    """Generate one complete synthetic incident."""
    latent_severity, profile_name = sample_latent_severity(priority)

    region, latitude, longitude = sample_region()
    population_density = sample_population_density(region)

    disaster_type = sample_disaster_type(region)
    incident_type = sample_incident_type(
        disaster_type,
        latent_severity,
    )

    infrastructure = generate_environment(
        disaster_type,
        incident_type,
        latent_severity,
    )

    human = generate_human_impact(
        latent_severity,
        profile_name,
        incident_type,
        population_density,
        infrastructure,
    )

    medical = generate_medical(
        priority,
        latent_severity,
        profile_name,
        human,
    )

    distances = generate_distances(
        region,
        latent_severity,
        infrastructure["road_status"],
    )

    return {
        "incident_id": f"UKD-{incident_number:06d}",
        "disaster_type": disaster_type,
        "incident_type": incident_type,
        **human,
        "water_level_m": infrastructure["water_level_m"],
        "flooded_area_percent": infrastructure["flooded_area_percent"],
        "road_status": infrastructure["road_status"],
        "house_damage": infrastructure["house_damage"],
        "evacuation_route_available": infrastructure[
            "evacuation_route_available"
        ],
        "power_available": infrastructure["power_available"],
        "communication_available": infrastructure[
            "communication_available"
        ],
        **medical,
        "latitude": latitude,
        "longitude": longitude,
        **distances,
        "population_density": population_density,
        "priority": priority,
    }


def generate_dataset():
    """Generate exactly 6,000 records with equal class counts."""
    records = []
    incident_number = 1

    for priority in PRIORITIES:
        for _ in range(RECORDS_PER_CLASS):
            records.append(
                generate_record(
                    incident_number=incident_number,
                    priority=priority,
                )
            )
            incident_number += 1

    dataframe = pd.DataFrame(records, columns=EXPECTED_COLUMNS)

    # Shuffle rows so priorities are not stored in contiguous blocks.
    dataframe = dataframe.sample(
        frac=1,
        random_state=RANDOM_SEED,
    ).reset_index(drop=True)

    return dataframe


# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------

def validate_dataset(df):
    """Programmatically verify all required data-quality constraints."""
    expected_total = RECORDS_PER_CLASS * len(PRIORITIES)

    assert list(df.columns) == EXPECTED_COLUMNS, (
        "Dataset columns do not match the required schema."
    )
    assert len(df) == expected_total, (
        f"Expected {expected_total} rows, found {len(df)}."
    )
    assert df["incident_id"].is_unique, "Duplicate incident_id detected."
    assert df.isnull().sum().sum() == 0, "Missing values detected."
    assert df.duplicated().sum() == 0, "Duplicate complete rows detected."

    # Check duplicates excluding the guaranteed-unique identifier.
    assert df.drop(columns=["incident_id"]).duplicated().sum() == 0, (
        "Duplicate feature/target rows detected."
    )

    class_counts = df["priority"].value_counts().to_dict()
    for priority in PRIORITIES:
        assert class_counts.get(priority, 0) == RECORDS_PER_CLASS, (
            f"{priority} is not exactly balanced."
        )

    allowed_values = {
        "disaster_type": set(DISASTER_TYPES),
        "incident_type": set(INCIDENT_TYPES),
        "road_status": set(ROAD_STATUSES),
        "house_damage": set(HOUSE_DAMAGE_LEVELS),
        "evacuation_route_available": {0, 1},
        "power_available": {0, 1},
        "communication_available": {0, 1},
        "medical_required": {0, 1},
        "medical_severity": set(MEDICAL_SEVERITIES),
        "ambulance_required": {0, 1},
        "priority": set(PRIORITIES),
    }

    for column, allowed in allowed_values.items():
        actual = set(df[column].unique())
        assert actual.issubset(allowed), (
            f"Unexpected values in {column}: {actual - allowed}"
        )

    numeric_ranges = {
        "people_affected": (0, 200),
        "people_trapped": (0, 100),
        "injured_people": (0, 30),
        "missing_people": (0, 30),
        "children_affected": (0, 50),
        "elderly_affected": (0, 40),
        "water_level_m": (0, 5),
        "flooded_area_percent": (0, 100),
        "latitude": (28.75, 31.45),
        "longitude": (77.55, 81.05),
        "nearest_hospital_km": (0.5, 30),
        "nearest_shelter_km": (0.2, 20),
        "nearest_rescue_team_km": (0.5, 30),
        "population_density": (50, 10_000),
    }

    for column, (minimum, maximum) in numeric_ranges.items():
        assert df[column].between(minimum, maximum).all(), (
            f"{column} contains values outside {minimum}–{maximum}."
        )

    numeric_df = df.select_dtypes(include=[np.number])

    assert np.isfinite(numeric_df.to_numpy()).all(), (
        "Infinite or non-finite numeric values detected."
    )
    assert (numeric_df >= 0).all().all(), "Negative values detected."

    # Internal human-impact consistency.
    assert (df["people_trapped"] <= df["people_affected"]).all()
    assert (df["injured_people"] <= df["people_affected"]).all()
    assert (df["missing_people"] <= df["people_affected"]).all()
    assert (df["children_affected"] <= df["people_affected"]).all()
    assert (df["elderly_affected"] <= df["people_affected"]).all()

    assert (
        df["children_affected"] + df["elderly_affected"]
        <= df["people_affected"]
    ).all()

    # Medical consistency.
    no_medical = df["medical_required"] == 0
    assert (df.loc[no_medical, "medical_severity"] == "None").all()
    assert (df.loc[no_medical, "ambulance_required"] == 0).all()

    medical = df["medical_required"] == 1
    assert (df.loc[medical, "medical_severity"] != "None").all()

    ambulance = df["ambulance_required"] == 1
    assert (df.loc[ambulance, "medical_required"] == 1).all()

    # Incident-type consistency.
    trapped_incidents = df["incident_type"] == "Trapped People"
    injured_incidents = df["incident_type"] == "Injured Person"
    missing_incidents = df["incident_type"] == "Missing Person"

    assert (df.loc[trapped_incidents, "people_trapped"] >= 1).all()
    assert (df.loc[injured_incidents, "injured_people"] >= 1).all()
    assert (df.loc[missing_incidents, "missing_people"] >= 1).all()

    # Confirm expected integer fields use integer-compatible dtypes.
    integer_columns = [
        "people_affected",
        "people_trapped",
        "injured_people",
        "missing_people",
        "children_affected",
        "elderly_affected",
        "evacuation_route_available",
        "power_available",
        "communication_available",
        "medical_required",
        "ambulance_required",
        "population_density",
    ]

    for column in integer_columns:
        assert pd.api.types.is_integer_dtype(df[column]), (
            f"{column} is not stored as an integer."
        )

    print("\nValidation completed successfully.")
    print("All schema, balance, range, uniqueness, and consistency checks passed.")


# ---------------------------------------------------------------------------
# Reporting and visualizations
# ---------------------------------------------------------------------------

def print_dataset_report(df):
    """Print the requested dataset report."""
    counts = df["priority"].value_counts()

    print("\n" + "=" * 72)
    print("DATASET GENERATION REPORT")
    print("=" * 72)

    print(f"\nTotal Records: {len(df):,}")
    print(f"P1 Count: {counts.get('P1', 0):,}")
    print(f"P2 Count: {counts.get('P2', 0):,}")
    print(f"P3 Count: {counts.get('P3', 0):,}")

    print(f"\nMissing Values: {int(df.isnull().sum().sum()):,}")
    print(f"Duplicate Rows: {int(df.duplicated().sum()):,}")
    print(
        "Duplicate Rows Excluding incident_id: "
        f"{int(df.drop(columns=['incident_id']).duplicated().sum()):,}"
    )

    print("\nClass Percentages:")
    print(
        df["priority"]
        .value_counts(normalize=True)
        .mul(100)
        .reindex(PRIORITIES)
        .round(2)
        .astype(str)
        .add("%")
        .to_string()
    )

    print("\nFeature Summary:")
    print(
        df.describe(include="all")
        .transpose()
        .to_string()
    )

    print("\nPriority-level Numeric Summary:")
    priority_summary = (
        df.groupby("priority")[
            [
                "people_affected",
                "people_trapped",
                "injured_people",
                "missing_people",
                "water_level_m",
                "flooded_area_percent",
                "nearest_rescue_team_km",
            ]
        ]
        .mean()
        .round(2)
        .reindex(PRIORITIES)
    )
    print(priority_summary.to_string())


def create_visualizations(df):
    """Generate the five requested visualizations."""
    sns.set_theme(style="whitegrid", context="notebook")

    palette = {
        "P1": "#d62728",
        "P2": "#ff8c00",
        "P3": "#f1c40f",
    }

    # 1. Class distribution
    plt.figure(figsize=(8, 5))
    ax = sns.countplot(
        data=df,
        x="priority",
        order=PRIORITIES,
        hue="priority",
        palette=palette,
        legend=False,
    )
    ax.set_title("Incident Priority Class Distribution")
    ax.set_xlabel("Priority")
    ax.set_ylabel("Number of incidents")

    for container in ax.containers:
        ax.bar_label(container, fmt="%d")

    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIRECTORY / "class_distribution.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.close()

    # 2. People affected
    plt.figure(figsize=(10, 6))
    sns.histplot(
        data=df,
        x="people_affected",
        hue="priority",
        hue_order=PRIORITIES,
        palette=palette,
        bins=40,
        element="step",
        stat="density",
        common_norm=False,
        alpha=0.28,
    )
    plt.title("Distribution of People Affected by Priority")
    plt.xlabel("People affected")
    plt.ylabel("Density")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIRECTORY / "people_affected_distribution.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.close()

    # 3. Injured people
    plt.figure(figsize=(10, 6))
    sns.histplot(
        data=df,
        x="injured_people",
        hue="priority",
        hue_order=PRIORITIES,
        palette=palette,
        discrete=True,
        multiple="layer",
        alpha=0.34,
    )
    plt.title("Distribution of Injured People by Priority")
    plt.xlabel("Injured people")
    plt.ylabel("Number of incidents")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIRECTORY / "injured_people_distribution.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.close()

    # 4. Disaster type distribution
    disaster_counts = (
        df.groupby(["disaster_type", "priority"])
        .size()
        .reset_index(name="count")
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=disaster_counts,
        x="disaster_type",
        y="count",
        hue="priority",
        hue_order=PRIORITIES,
        palette=palette,
    )
    plt.title("Disaster Type Distribution by Priority")
    plt.xlabel("Disaster type")
    plt.ylabel("Number of incidents")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIRECTORY / "disaster_type_distribution.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.close()

    # 5. Correlation matrix
    correlation_data = df.copy()

    ordinal_mappings = {
        "road_status": {
            "Open": 0,
            "Partially Blocked": 1,
            "Blocked": 2,
        },
        "house_damage": {
            "None": 0,
            "Minor": 1,
            "Moderate": 2,
            "Severe": 3,
        },
        "medical_severity": {
            "None": 0,
            "Low": 1,
            "Moderate": 2,
            "Critical": 3,
        },
        "priority": {
            "P3": 0,
            "P2": 1,
            "P1": 2,
        },
    }

    for column, mapping in ordinal_mappings.items():
        correlation_data[column] = correlation_data[column].map(mapping)

    correlation_columns = [
        "people_affected",
        "people_trapped",
        "injured_people",
        "missing_people",
        "children_affected",
        "elderly_affected",
        "water_level_m",
        "flooded_area_percent",
        "road_status",
        "house_damage",
        "evacuation_route_available",
        "power_available",
        "communication_available",
        "medical_required",
        "medical_severity",
        "ambulance_required",
        "nearest_hospital_km",
        "nearest_shelter_km",
        "nearest_rescue_team_km",
        "population_density",
        "priority",
    ]

    correlation_matrix = correlation_data[correlation_columns].corr()

    plt.figure(figsize=(18, 14))
    sns.heatmap(
        correlation_matrix,
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        square=False,
        linewidths=0.25,
        cbar_kws={"shrink": 0.8},
    )
    plt.title("Numeric and Ordinal Feature Correlation Matrix")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIRECTORY / "correlation_matrix.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.close()

    print("\nVisualizations generated:")
    print("  class_distribution.png")
    print("  people_affected_distribution.png")
    print("  injured_people_distribution.png")
    print("  disaster_type_distribution.png")
    print("  correlation_matrix.png")


# ---------------------------------------------------------------------------
# Main execution
# ---------------------------------------------------------------------------

def main():
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

    print("Generating synthetic Uttarakhand disaster incidents...")

    dataset = generate_dataset()
    validate_dataset(dataset)

    output_path = OUTPUT_DIRECTORY / OUTPUT_FILE
    dataset.to_csv(output_path, index=False)

    print_dataset_report(dataset)
    create_visualizations(dataset)

    print("\n" + "=" * 72)
    print("OUTPUT")
    print("=" * 72)
    print(f"CSV saved to: {output_path.resolve()}")
    print(f"Random seed: {RANDOM_SEED}")

    print("\nML readiness:")
    print("CSV")
    print("  ↓")
    print("Preprocessing")
    print("  ↓")
    print("Categorical Encoding")
    print("  ↓")
    print("Train/Test Split")
    print("  ↓")
    print("Random Forest / XGBoost")
    print("  ↓")
    print("P1 / P2 / P3 Prediction")

    print("\nDataset is ready for incident-priority classification.")


if __name__ == "__main__":
    main()


Generating synthetic Uttarakhand disaster incidents...

Validation completed successfully.
All schema, balance, range, uniqueness, and consistency checks passed.

DATASET GENERATION REPORT

Total Records: 6,000
P1 Count: 2,000
P2 Count: 2,000
P3 Count: 2,000

Missing Values: 0
Duplicate Rows: 0
Duplicate Rows Excluding incident_id: 0

Class Percentages:
priority
P1    33.33%
P2    33.33%
P3    33.33%

Feature Summary:
                             count unique                top  freq       mean          std        min        25%        50%        75%        max
incident_id                   6000   6000         UKD-001783     1        NaN          NaN        NaN        NaN        NaN        NaN        NaN
disaster_type                 6000      4              Flood  2076        NaN          NaN        NaN        NaN        NaN        NaN        NaN
incident_type                 6000      8     Trapped People  1024        NaN          NaN        NaN        NaN        NaN        NaN      

In [ ]:
from sklearn.preprocessing import 